# Wikipedia knowledge graph + Text2Cypher — interactive tour

The **LangChain-forward** demo. An LLM (`LLMGraphTransformer`) built this graph
from raw Wikipedia article leads; natural-language questions are answered by
**generating Cypher** through an LCEL pipeline.

**Prerequisite — build the graph first:**

```bash
cd langchain
WIKI_LIMIT=200 .venv/bin/python examples/demos/02_wikipedia_kg/build_kg.py
```

In [1]:
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../02_wikipedia_kg
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for `_common`

import pandas as pd
from _common import agens, config
from _common.models import get_llm
from langchain_agensgraph import AgensCypherQAChain

GRAPH = "wikipedia_kg"
# enhanced_schema=True -> the schema carries example property values, which the
# LLM uses to write better Cypher.
graph = agens.make_graph(GRAPH, create=False, enhanced_schema=True)
print("connected to", config.url().split("@")[-1])

connected to localhost:55432/agensgraph_demos


## The knowledge graph the LLM built

Typed entities + LLM-named relationships, plus `Document` provenance nodes
(`(:Document)-[:MENTIONS]->(entity)`).

In [2]:
n = graph.query("MATCH (n) RETURN count(n) AS c")[0]["c"]
e = graph.query("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
print(f"{n:,} nodes · {e:,} edges")
pd.DataFrame(graph.query(
    "MATCH (n) RETURN label(n) AS label, count(*) AS count ORDER BY count DESC"
))

7,087 nodes · 12,344 edges


,label,count
0,Person,1417
1,Location,1270
2,Concept,1241
3,Organization,689
4,Work,668
5,Document,498
6,Event,486
7,Group,346
8,Field,306
9,Technology,133


In [3]:
# A peek at some extracted relationships (excluding the Document->entity
# provenance edges, so we see entity-to-entity relationships).
pd.DataFrame(graph.query(
    "MATCH (a)-[r]->(b) WHERE type(r) <> 'MENTIONS' "
    "RETURN label(a) AS from_type, a.id AS source, type(r) AS rel, b.id AS target LIMIT 12"
))

,from_type,source,rel,target
0,Concept,Anarchism,RELATED_TO,libertarian socialism
1,Concept,Anarchism,RELATED_TO,anti-capitalist movement
2,Concept,Anarchism,RELATED_TO,anti-war movement
3,Concept,Anarchism,RELATED_TO,anti-globalisation movement
4,Concept,solar radiation,RELATED_TO,atmospheric composition
5,Concept,solar radiation,RELATED_TO,geographic location
6,Concept,solar radiation,RELATED_TO,time
7,Concept,solar radiation,RELATED_TO,visible light
8,Concept,Ice–albedo feedback,RELATED_TO,ice caps
9,Concept,Ice–albedo feedback,RELATED_TO,glaciers


## What the Text2Cypher model sees

The graph's schema (labels, properties, example values, relationship patterns) is
fed to the LLM so it writes valid AgensGraph Cypher.

In [4]:
print(graph.get_schema.strip()[:1500])

Node properties are the following:
        [{'labels': 'Concept', 'properties': [{'property': 'id', 'type': 'string', 'examples': ['Anarchism', 'libertarian socialism', 'anti-capitalist movement']}]}, {'labels': 'Event', 'properties': [{'property': 'id', 'type': 'string', 'examples': ['Enlightenment', 'Paris Commune', 'Russian Civil War']}]}, {'labels': 'Document', 'properties': [{'property': 'id', 'type': 'string', 'examples': ['3217bea3396440f7861e71ca5cc463bc', '7ac2c48e1665150e83d8c057640d598b', '9b4fc14acac8ed05e3a01c53102fbc4d']}, {'property': 'source', 'type': 'string', 'examples': ['wikipedia']}, {'property': 'title', 'type': 'string', 'examples': ['Anarchism', 'Albedo', 'A']}, {'property': 'url', 'type': 'string', 'examples': ['https://en.wikipedia.org/wiki/Anarchism', 'https://en.wikipedia.org/wiki/Albedo', 'https://en.wikipedia.org/wiki/A']}]}, {'labels': 'Location', 'properties': [{'property': 'id', 'type': 'string', 'examples': ['Earth', 'Alabama', 'United States']}]}, {'l

## Ask in natural language (Text2Cypher)

`AgensCypherQAChain` composes the pipeline and carries the AgensGraph dialect rules:

```python
question -> schema + question -> llm -> cypher
         -> quote labels, refuse writes, EXPLAIN
         -> run inside a read-only transaction
         -> rows -> llm -> answer
```

`return_intermediate_steps=True` is what makes the generated Cypher visible below.


In [5]:
chain = AgensCypherQAChain.from_llm(
    get_llm(),
    graph=graph,
    top_k=25,
    return_intermediate_steps=True,
    # What refuses a generated write is a read-only transaction, and one is not a
    # boundary for a role that may run a command on the server's host -- which the
    # role these demos connect as can, being the one that created the databases.
    # The refusal is accepted here so the notebook runs; an application serving
    # questions from the public should connect as a role that cannot, and leave
    # this alone.
    allow_server_programs=True,
)

def run(question):
    out = chain.invoke({"query": question})
    cypher, rows = out["intermediate_steps"]
    print("Q:", question)
    print("\nGenerated Cypher:\n  " + cypher["query"].replace("\n", "\n  "))
    print(f"\nRows: {len(rows['context'])}")
    print("\nAnswer:\n" + out["result"])

run("What types of entities are in the graph, and how many of each?")


Q: What types of entities are in the graph, and how many of each?

Generated Cypher:
  MATCH (n) RETURN label(n) AS label, count(*) AS n ORDER BY n DESC LIMIT 25

Rows: 11

Answer:
The graph contains the following types of entities and their counts:

- Person: 1417
- Location: 1270
- Concept: 1241
- Organization: 689
- Work: 668
- Document: 498
- Event: 486
- Group: 346
- Field: 306
- Technology: 133
- Award: 33


In [6]:
run("Which 5 people are connected to the most other entities?")

Q: Which 5 people are connected to the most other entities?

Generated Cypher:
  MATCH (p:"Person")-[r]->(e) RETURN p."id" AS personId, count(*) AS connections ORDER BY connections DESC LIMIT 5

Rows: 5

Answer:
The 5 people connected to the most other entities are:
1. Arthur Aikin - 26 connections
2. Ajax - 24 connections
3. Alfons Maria Jakob - 23 connections
4. Alain Connes - 22 connections
5. Andrei Tarkovsky - 20 connections


In [7]:
run("List 5 organizations in the graph and one entity each is connected to.")

Q: List 5 organizations in the graph and one entity each is connected to.

Generated Cypher:
  MATCH (o:"Organization")-[r]->(e) RETURN o.id AS organization, e.id AS connectedEntity LIMIT 5

Rows: 5

Answer:
1. Academy of Motion Picture Arts and Sciences - connected to Beverly Hills
2. Lighting company - connected to Allan Dwan
3. Regency of Algiers - connected to French
4. Regency of Algiers - connected to Mediterranean Sea
5. Regency of Algiers - connected to Algiers


## What you can do with this

- **Build a graph from unstructured text** with one LangChain component
  (`LLMGraphTransformer`) — typed entities + relationships, structured output.
- **Query it in natural language** — the LLM writes schema-grounded, read-only
  Cypher; results ground the final answer. All composed with LCEL.

Try your own:

```python
run("your question here")
```

When finished, close the shared pool: `agens.close()`